In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import contextlib
import logging
import os
import numpy as np
import torch
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import backbones
import common
import metrics
import patchcore
import sampler
import utils
import mvtec as mvtec_dataset
%reload_ext autoreload

logging.basicConfig(level=logging.INFO)
LOGGER = logging.getLogger(__name__)
# adapted from https://github.com/amazon-science/patchcore-inspection/blob/main/bin/run_patchcore.py 
def run_patchcore(
    data_path,
    subdataset,
    batch_size=2,
    resize=256,
    imagesize=224,
    num_workers=8,
    train_val_split=1.0,
    augment=False,
    backbone_name="wideresnet50",
    layers_to_extract_from=["layer2", "layer3"],
    pretrain_embed_dimension=1024,
    target_embed_dimension=1024,
    patchsize=3,
    patchscore="max",
    patchoverlap=0.0,
    anomaly_scorer_num_nn=5,
    sampler_name="approx_greedy_coreset",
    percentage=0.1,
    gpu_id=0,
    seed=0,
    save_segmentation_images=False,
    save_patchcore_model=True,
    results_path="results",
    log_project="patchcore-project",
    log_group="patchcore-project"
    )  :
    run_save_path = utils.create_storage_folder(
        results_path, log_project, log_group, mode="iterate"
    )
    
    device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    LOGGER.info(f"Using device: {device}")
    
    device_context = (
        torch.cuda.device(f"cuda:{gpu_id}")
        if "cuda" in device.type.lower()
        else contextlib.suppress()
    )
    
    utils.fix_seeds(seed, device)
    
    train_dataset = mvtec_dataset.MVTecDataset(
        data_path,
        classname=subdataset,
        resize=resize,
        train_val_split=train_val_split,
        imagesize=imagesize,
        split=mvtec_dataset.DatasetSplit.TRAIN,
        seed=seed,
        augment=augment,
    )

    test_dataset = mvtec_dataset.MVTecDataset(
        data_path,
        classname=subdataset,
        resize=resize,
        imagesize=imagesize,
        split=mvtec_dataset.DatasetSplit.TEST,
        seed=seed,
    )

    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    train_dataloader.name = f"mvtec_{subdataset}"
    
    dataloaders = {
        "training": train_dataloader,
        "testing": test_dataloader,
    }
    
    if sampler_name == "identity":
        sampler = patchcore.sampler.IdentitySampler()
    elif sampler_name == "greedy_coreset":
        sampler =  patchcore.sampler.GreedyCoresetSampler(percentage, device)
    elif sampler_name == "approx_greedy_coreset":
        sampler =  patchcore.sampler.ApproximateGreedyCoresetSampler(percentage, device)
    elif sampler_name == "KNN_coreset":
        sampler =  patchcore.sampler.FastKNNSampler(percentage, device)
    
    with device_context:
        torch.cuda.empty_cache()
        
        backbone = backbones.load(backbone_name)
        backbone.name, backbone.seed = backbone_name, None

        nn_method = common.FaissNN(True, 8) 
        
        patchcore_instance = patchcore.PatchCore(device)
        patchcore_instance.load(
            backbone=backbone,
            layers_to_extract_from=layers_to_extract_from,
            device=device,
            input_shape=(3, imagesize, imagesize),
            pretrain_embed_dimension=pretrain_embed_dimension,
            target_embed_dimension=target_embed_dimension,
            patchsize=patchsize,
            featuresampler=sampler,
            anomaly_scorer_num_nn=anomaly_scorer_num_nn,
            nn_method=nn_method,
        )

        LOGGER.info("Training PatchCore model...")
        patchcore_instance.fit(dataloaders["training"])

        LOGGER.info("Testing PatchCore model...")
        scores, segmentations, labels_gt, masks_gt = patchcore_instance.predict(
            dataloaders["testing"]
        )

        anomaly_labels = [
            x[1] != "good" for x in dataloaders["testing"].dataset.data_to_iterate
        ]
        
        auroc = metrics.compute_imagewise_retrieval_metrics(
            scores, anomaly_labels
        )["auroc"]
        

        full_pixel_auroc = metrics.compute_pixelwise_retrieval_metrics(
            segmentations, masks_gt
        )["auroc"]
        
        sel_idxs = []
        for i in range(len(masks_gt)):
            if np.sum(masks_gt[i]) > 0:
                sel_idxs.append(i)
                
        anomaly_pixel_auroc = 0
        if len(sel_idxs) > 0:
            anomaly_pixel_auroc = metrics.compute_pixelwise_retrieval_metrics(
                [segmentations[i] for i in sel_idxs],
                [masks_gt[i] for i in sel_idxs],
            )["auroc"]
        
        results_dict = {
            "dataset_name": subdataset,
            "instance_auroc": auroc,
            "full_pixel_auroc": full_pixel_auroc,
            "anomaly_pixel_auroc": anomaly_pixel_auroc,
            "raw_scores": scores,
            "raw_segmentations": segmentations,
            "raw_labels": [x[1] for x in dataloaders["testing"].dataset.data_to_iterate],
            "raw_image_paths": [x[2] for x in dataloaders["testing"].dataset.data_to_iterate]
        }
        
        for key, item in {
            "instance_auroc": auroc,
            "full_pixel_auroc": full_pixel_auroc,
            "anomaly_pixel_auroc": anomaly_pixel_auroc
        }.items():
            LOGGER.info("{0}: {1:3.3f}".format(key, item))
        
        if save_segmentation_images:
            image_paths = [
                x[2] for x in dataloaders["testing"].dataset.data_to_iterate
            ]
            mask_paths = [
                x[3] for x in dataloaders["testing"].dataset.data_to_iterate
            ]

            def image_transform(image):
                in_std = np.array(
                    dataloaders["testing"].dataset.transform_std
                ).reshape(-1, 1, 1)
                in_mean = np.array(
                    dataloaders["testing"].dataset.transform_mean
                ).reshape(-1, 1, 1)
                image = dataloaders["testing"].dataset.transform_img(image)
                return np.clip(
                    (image.numpy() * in_std + in_mean) * 255, 0, 255
                ).astype(np.uint8)

            def mask_transform(mask):
                return dataloaders["testing"].dataset.transform_mask(mask).numpy()

            image_save_path = os.path.join(
                run_save_path, "segmentation_images", subdataset
            )
            os.makedirs(image_save_path, exist_ok=True)
            utils.plot_segmentation_images(
                image_save_path,
                image_paths,
                segmentations,
                scores,
                mask_paths,
                image_transform=image_transform,
                mask_transform=mask_transform,
            )
        if save_patchcore_model:
            patchcore_save_path = os.path.join(
                run_save_path, "models", subdataset
            )
            os.makedirs(patchcore_save_path, exist_ok=True)
            patchcore_instance.save_to_path(patchcore_save_path)
        
        return results_dict


In [ ]:
def extract_scores(results):
 
    scores = results["raw_scores"]
    labels = results["raw_labels"]
    image_paths = results["raw_image_paths"]
    
    return scores, labels, image_paths


In [ ]:
%autoreload 2
results_front = run_patchcore(
    data_path= r"C:\Thesis\Data\paperclips_sorted",
    subdataset="paperclips_front",
    batch_size=2,
    backbone_name="wideresnet50",
    sampler_name="approx_greedy_coreset",
    gpu_id=0,
    save_segmentation_images=False,
    save_patchcore_model=True,  
    results_path=r"C:\Thesis\Models\patchcore\checkpoints" 
)

results_side = run_patchcore(
    data_path= r"C:\Thesis\Data\paperclips_sorted",
    subdataset="paperclips_side",
    batch_size=2,
    backbone_name="wideresnet50",
    sampler_name="approx_greedy_coreset",
    gpu_id=0,
    save_segmentation_images=False,
    save_patchcore_model=True,
    results_path=r"C:\Thesis\Models\patchcore\checkpoints" 
)

INFO:__main__:Using device: cuda:0
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torch\utils\data\dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 

Type of ground_truth_masks: <class 'list'>
Length of ground_truth_masks: 98
Type of mask at index 0: <class 'list'>
  List length: 1
Type of mask at index 1: <class 'list'>
  List length: 1
Type of mask at index 2: <class 'list'>
  List length: 1
Type of mask at index 3: <class 'list'>
  List length: 1
Type of mask at index 4: <class 'list'>
  List length: 1
Type of mask at index 5: <class 'list'>
  List length: 1
Type of mask at index 6: <class 'list'>
  List length: 1
Type of mask at index 7: <class 'list'>
  List length: 1
Type of mask at index 8: <class 'list'>
  List length: 1
Type of mask at index 9: <class 'list'>
  List length: 1
Type of mask at index 10: <class 'list'>
  List length: 1
Type of mask at index 11: <class 'list'>
  List length: 1
Type of mask at index 12: <class 'list'>
  List length: 1
Type of mask at index 13: <class 'list'>
  List length: 1
Type of mask at index 14: <class 'list'>
  List length: 1
Type of mask at index 15: <class 'list'>
  List length: 1
Type o

INFO:__main__:instance_auroc: 0.995
INFO:__main__:full_pixel_auroc: 0.950
INFO:__main__:anomaly_pixel_auroc: 0.934
INFO:patchcore:Saving PatchCore data.
INFO:__main__:Using device: cuda:0
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torch\utils\data\dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\feder\anaconda3\envs\patchc

KeyboardInterrupt: 

In [ ]:
print("front keys", results_front.keys())
print("side keys:", results_side.keys())

Available keys in results_front: dict_keys(['dataset_name', 'instance_auroc', 'full_pixel_auroc', 'anomaly_pixel_auroc', 'raw_scores', 'raw_segmentations', 'raw_labels', 'raw_image_paths'])
Available keys in results_side: dict_keys(['dataset_name', 'instance_auroc', 'full_pixel_auroc', 'anomaly_pixel_auroc', 'raw_scores', 'raw_segmentations', 'raw_labels', 'raw_image_paths'])


In [ ]:

def late_fusion_scores(front_results, side_results, method='max', output_dir="fusion_results"):
    

    LOGGER = logging.getLogger(__name__)
    
    os.makedirs(output_dir, exist_ok=True)

    front_scores = front_results["raw_scores"]
    side_scores = side_results["raw_scores"]
    front_labels = front_results["raw_labels"]
    side_labels = side_results["raw_labels"]
    front_paths = front_results["raw_image_paths"]
    side_paths = side_results["raw_image_paths"]
    
    LOGGER.info(f"Front scores: {len(front_scores)}, Side scores: {len(side_scores)}")
    LOGGER.info(f"Front labels: {len(front_labels)}, Side labels: {len(side_labels)}")
    
    front_label_counts = pd.Series(front_labels).value_counts()
    side_label_counts = pd.Series(side_labels).value_counts()
    LOGGER.info(f"Front label distribution: {dict(front_label_counts)}")
    LOGGER.info(f"Side label distribution: {dict(side_label_counts)}")
    
    front_files = [os.path.basename(path) for path in front_paths]
    side_files = [os.path.basename(path) for path in side_paths]
    
    front_df = pd.DataFrame({
        'filename': front_files,
        'label': front_labels,
        'score': front_scores,
        'view': 'front',
        'original_index': range(len(front_files))  
    })
    
    side_df = pd.DataFrame({
        'filename': side_files,
        'label': side_labels,
        'score': side_scores,
        'view': 'side',
        'original_index': range(len(side_files))  
    })
    

    def extract_identifier(filename):
        parts = filename.split('_')
        if len(parts) > 1:
            return parts[0]
        return None
    
    front_df['identifier'] = front_df['filename'].apply(extract_identifier)
    side_df['identifier'] = side_df['filename'].apply(extract_identifier)
    
    front_df = front_df[front_df['identifier'].notna()]
    side_df = side_df[side_df['identifier'].notna()]
    
    front_df.to_csv(os.path.join(output_dir, "front_pre_merge.csv"), index=False)
    side_df.to_csv(os.path.join(output_dir, "side_pre_merge.csv"), index=False)

    front_duplicates = front_df['identifier'].duplicated(keep=False)
    side_duplicates = side_df['identifier'].duplicated(keep=False)
    
    front_dup_count = front_duplicates.sum()
    side_dup_count = side_duplicates.sum()
    
    LOGGER.info(f"Duplicate identifiers: Front={front_dup_count}, Side={side_dup_count}")
    
    if front_dup_count > 0:
        front_dup_df = front_df[front_duplicates]
        front_dup_labels = front_dup_df['label'].value_counts()
        LOGGER.info(f"Front duplicates by label: {dict(front_dup_labels)}")
    
        sample_dups = front_dup_df['identifier'].unique()[:5]
        LOGGER.info(f"Sample duplicate identifiers: {sample_dups}")
        
        for dup_id in sample_dups:
            dup_rows = front_df[front_df['identifier'] == dup_id]
            LOGGER.info(f"Files with identifier '{dup_id}': {dup_rows['filename'].tolist()}")
    
    if front_dup_count > 0:
        front_df['identifier'] = front_df.apply(
            lambda row: f"{row['identifier']}_{row['original_index']}" if front_duplicates[row.name] else row['identifier'], 
            axis=1
        )
        side_df['identifier'] = side_df.apply(
            lambda row: f"{row['identifier']}_{row['original_index']}" if side_duplicates[row.name] else row['identifier'], 
            axis=1
        )

    front_ids = set(front_df['identifier'])
    side_ids = set(side_df['identifier'])
    common_ids = front_ids.intersection(side_ids)
    
    LOGGER.info(f"Common identifiers after adjustment: {len(common_ids)}")

    front_common = front_df[front_df['identifier'].isin(common_ids)]
    front_common_labels = front_common['label'].value_counts()
    LOGGER.info(f"Common samples label distribution: {dict(front_common_labels)}")
 
    merged_df = pd.merge(
        front_df, 
        side_df, 
        on='identifier', 
        suffixes=('_front', '_side'),
        how='inner'
    )

    is_anomaly = (merged_df['label_front'] != 'good').astype(int)
    unique_classes = np.unique(is_anomaly)
    
    if len(unique_classes) < 2 or is_anomaly.sum() < 2 or (~is_anomaly).sum() < 2:
        LOGGER.warning("Insufficient class balance in merged data. Forcing inclusion of both classes.")

        front_normal = front_df[front_df['label'] == 'good']
        front_anomaly = front_df[front_df['label'] != 'good']

        side_normal = side_df[side_df['label'] == 'good']
        side_anomaly = side_df[side_df['label'] != 'good']
        
        LOGGER.info(f"Front normal: {len(front_normal)}, Front anomaly: {len(front_anomaly)}")
        LOGGER.info(f"Side normal: {len(side_normal)}, Side anomaly: {len(side_anomaly)}")

        min_normal = min(len(front_normal), len(side_normal))
        min_anomaly = min(len(front_anomaly), len(side_anomaly))
      
        front_normal_sample = front_normal.iloc[:min_normal].copy()
        front_anomaly_sample = front_anomaly.iloc[:min_anomaly].copy()
        side_normal_sample = side_normal.iloc[:min_normal].copy()
        side_anomaly_sample = side_anomaly.iloc[:min_anomaly].copy()
        
        front_normal_sample['forced_id'] = [f"normal_{i}" for i in range(min_normal)]
        front_anomaly_sample['forced_id'] = [f"anomaly_{i}" for i in range(min_anomaly)]
        side_normal_sample['forced_id'] = [f"normal_{i}" for i in range(min_normal)]
        side_anomaly_sample['forced_id'] = [f"anomaly_{i}" for i in range(min_anomaly)]
        
        front_combined = pd.concat([front_normal_sample, front_anomaly_sample])
        side_combined = pd.concat([side_normal_sample, side_anomaly_sample])

        merged_df = pd.merge(
            front_combined, 
            side_combined, 
            on='forced_id', 
            suffixes=('_front', '_side')
        )
        
        LOGGER.info(f"Forced pairing created {len(merged_df)} samples with balanced classes")
    
    if len(merged_df) < 2:
        LOGGER.error(f"Critical: Only {len(merged_df)} matches found between views. Cannot proceed.")
        
        dummy_result = {
            'front_results': front_results,
            'side_results': side_results,
            'fusion_results': {
                'front_auroc': 0.5,
                'side_auroc': 0.5,
                f'fused_{method}_auroc': 0.5,
                'num_samples': 0,
                'fusion_method': method,
                'error': 'No matches found between views'
            },
            'merged_df': merged_df
        }
        return dummy_result
    
    LOGGER.info(f"Successfully matched {len(merged_df)} images between front and side views")

    if method == 'max':
        merged_df['fused_score'] = merged_df[['score_front', 'score_side']].max(axis=1)
    elif method == 'min':
        merged_df['fused_score'] = merged_df[['score_front', 'score_side']].min(axis=1)
    elif method == 'mean':
        merged_df['fused_score'] = merged_df[['score_front', 'score_side']].mean(axis=1)
    elif method == 'product':
        merged_df['fused_score'] = merged_df['score_front'] * merged_df['score_side']
    else:
        raise ValueError(f"Unknown fusion method: {method}")
   
    merged_df.to_csv(os.path.join(output_dir, f"fused_scores_{method}.csv"), index=False)
    
    is_anomaly = (merged_df['label_front'] != 'good').astype(int)

    normal_count = (~is_anomaly).sum()
    anomaly_count = is_anomaly.sum()
    LOGGER.info(f"Final class balance - Normal: {normal_count}, Anomaly: {anomaly_count}")
    
    if normal_count > 0 and anomaly_count > 0:

        front_auroc = roc_auc_score(is_anomaly, merged_df['score_front'])
        side_auroc = roc_auc_score(is_anomaly, merged_df['score_side'])
        fused_auroc = roc_auc_score(is_anomaly, merged_df['fused_score'])
    else:
        LOGGER.error("Cannot calculate AUROC: Need both normal and anomaly samples")
        front_auroc = side_auroc = fused_auroc = 0.5
    
    orig_front_is_anomaly = (np.array(front_labels) != 'good').astype(int)
    orig_side_is_anomaly = (np.array(side_labels) != 'good').astype(int)
    
    orig_front_auroc = roc_auc_score(orig_front_is_anomaly, front_scores)
    orig_side_auroc = roc_auc_score(orig_side_is_anomaly, side_scores)
    
    LOGGER.info(f"Original Front AUROC: {orig_front_auroc:.4f}")
    LOGGER.info(f"Original Side AUROC: {orig_side_auroc:.4f}")

    fusion_results = {
        'front_auroc': front_auroc,
        'side_auroc': side_auroc,
        f'fused_{method}_auroc': fused_auroc,
        'num_samples': len(merged_df),
        'fusion_method': method,
        'original_front_auroc': orig_front_auroc,
        'original_side_auroc': orig_side_auroc,
        'normal_count': normal_count,
        'anomaly_count': anomaly_count
    }
    
    with open(os.path.join(output_dir, f"fusion_results_{method}.txt"), 'w') as f:
        f.write("Late Fusion Results Summary\n")
        f.write(f"Fusion method: {method}\n")
        f.write(f"Number of matched samples: {len(merged_df)}\n")
        f.write(f"Class balance - Normal: {normal_count}, Anomaly: {anomaly_count}\n\n")
        f.write(f"Original Front view AUROC: {orig_front_auroc:.4f}\n")
        f.write(f"Original Side view AUROC: {orig_side_auroc:.4f}\n\n")
        f.write(f"Merged dataset Front view AUROC: {front_auroc:.4f}\n")
        f.write(f"Merged dataset Side view AUROC: {side_auroc:.4f}\n")
        f.write(f"Fused AUROC: {fused_auroc:.4f}\n\n")
    
        improvement_front = (fused_auroc - front_auroc) * 100
        improvement_side = (fused_auroc - side_auroc) * 100
        
        f.write(f"Improvement over front view: {improvement_front:.2f}%\n")
        f.write(f"Improvement over side view: {improvement_side:.2f}%\n")
    
    LOGGER.info(f"Fusion complete! Results saved to {output_dir}")
    
    return {
        'front_results': front_results,
        'side_results': side_results,
        'fusion_results': fusion_results,
        'merged_df': merged_df
    }

In [ ]:
def analyze_fusion_results(front_results, side_results, fusion_results, output_dir="fusion_analysis"):

    os.makedirs(output_dir, exist_ok=True)
    
    merged_df = fusion_results['merged_df']
    
    plt.figure(figsize=(15, 8))
    plt.subplot(2, 2, 1)
    sns.histplot(data=merged_df, x='score_front', hue='label_front', 
                 kde=True, palette='Set1', bins=20)
    plt.title('Front View Score Distribution')
    plt.xlabel('Anomaly Score')
    plt.ylabel('Count')
    
    plt.subplot(2, 2, 2)
    sns.histplot(data=merged_df, x='score_side', hue='label_side', 
                 kde=True, palette='Set1', bins=20)
    plt.title('Side View Score Distribution')
    plt.xlabel('Anomaly Score')
    plt.ylabel('Count')
    
    plt.subplot(2, 2, 3)
    sns.histplot(data=merged_df, x='fused_score', hue='label_front', 
                 kde=True, palette='Set1', bins=20)
    plt.title(f"Fused Scores Distribution ({fusion_results['fusion_results']['fusion_method']} method)")
    plt.xlabel('Anomaly Score')
    plt.ylabel('Count')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "score_distributions.png"), dpi=300)
    plt.close()
    
    plt.figure(figsize=(10, 8))
    
    is_anomaly = (merged_df['label_front'] != 'good')
    colors = ['blue' if not a else 'red' for a in is_anomaly]
    
    plt.scatter(merged_df['score_front'], merged_df['score_side'], c=colors, alpha=0.6)
    plt.title('Front vs Side View Anomaly Scores')
    plt.xlabel('Front View Score')
    plt.ylabel('Side View Score')
    plt.grid(True, alpha=0.3)
    
    plt.scatter([], [], c='blue', label='Normal')
    plt.scatter([], [], c='red', label='Anomaly')
    plt.legend()
    
    max_val = max(merged_df['score_front'].max(), merged_df['score_side'].max())
    plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
    
    plt.savefig(os.path.join(output_dir, "front_vs_side_scores.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(10, 8))
    
    y_true = is_anomaly.astype(int)

    fpr_front, tpr_front, _ = roc_curve(y_true, merged_df['score_front'])
    fpr_side, tpr_side, _ = roc_curve(y_true, merged_df['score_side'])
    fpr_fused, tpr_fused, _ = roc_curve(y_true, merged_df['fused_score'])
    
    roc_auc_front = auc(fpr_front, tpr_front)
    roc_auc_side = auc(fpr_side, tpr_side)
    roc_auc_fused = auc(fpr_fused, tpr_fused)

    plt.plot(fpr_front, tpr_front, color='blue', lw=2, 
             label=f'Front View (AUC = {roc_auc_front:.4f})')
    plt.plot(fpr_side, tpr_side, color='green', lw=2, 
             label=f'Side View (AUC = {roc_auc_side:.4f})')
    plt.plot(fpr_fused, tpr_fused, color='red', lw=2, 
             label=f'Fused {fusion_results["fusion_results"]["fusion_method"]} (AUC = {roc_auc_fused:.4f})')

    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    
    plt.savefig(os.path.join(output_dir, "roc_curves.png"), dpi=300)
    plt.close()
    
    plt.figure(figsize=(10, 6))
    
    method = fusion_results['fusion_results']['fusion_method']
    fusion_auroc_key = f'fused_{method}_auroc'
    
    methods = ['Front', 'Side', f'Fused ({method})']
    auroc_values = [
        fusion_results['fusion_results']['front_auroc'],
        fusion_results['fusion_results']['side_auroc'],
        fusion_results['fusion_results'][fusion_auroc_key]
    ]
    
    if 'original_front_auroc' in fusion_results['fusion_results']:
        methods.extend(['Original Front', 'Original Side'])
        auroc_values.extend([
            fusion_results['fusion_results']['original_front_auroc'],
            fusion_results['fusion_results']['original_side_auroc']
        ])
    
    plt.bar(methods, auroc_values, color=['blue', 'green', 'red', 'lightblue', 'lightgreen'])
    plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
    plt.title('AUROC Comparison')
    plt.ylabel('AUROC')
    plt.ylim([0, 1.1])
    
    for i, v in enumerate(auroc_values):
        plt.text(i, v + 0.02, f'{v:.4f}', ha='center')
    
    plt.savefig(os.path.join(output_dir, "auroc_comparison.png"), dpi=300)
    plt.close()
    
    plt.figure(figsize=(12, 8))
    
    boxplot_data = pd.DataFrame({
        'Front (Normal)': merged_df.loc[~is_anomaly, 'score_front'],
        'Front (Anomaly)': merged_df.loc[is_anomaly, 'score_front'],
        'Side (Normal)': merged_df.loc[~is_anomaly, 'score_side'],
        'Side (Anomaly)': merged_df.loc[is_anomaly, 'score_side'],
        f'Fused {method} (Normal)': merged_df.loc[~is_anomaly, 'fused_score'],
        f'Fused {method} (Anomaly)': merged_df.loc[is_anomaly, 'fused_score']
    })
    
    sns.boxplot(data=boxplot_data)
    plt.title('Score Distribution by Class')
    plt.ylabel('Anomaly Score')
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_dir, "score_boxplots.png"), dpi=300)
    plt.close()
    
    with open(os.path.join(output_dir, "analysis_summary.txt"), 'w') as f:
        f.write("PatchCore Late Fusion Analysis Summary\n")
        f.write(f"Fusion method: {method}\n")
        f.write(f"Number of matched samples: {len(merged_df)}\n\n")
        
        f.write("AUROC Values:\n")
        f.write(f"- Front view: {fusion_results['fusion_results']['front_auroc']:.4f}\n")
        f.write(f"- Side view: {fusion_results['fusion_results']['side_auroc']:.4f}\n")
        f.write(f"- Fused ({method}): {fusion_results['fusion_results'][fusion_auroc_key]:.4f}\n\n")
        
        if 'original_front_auroc' in fusion_results['fusion_results']:
            f.write("Original AUROC Values (pre-matching):\n")
            f.write(f"- Front view: {fusion_results['fusion_results']['original_front_auroc']:.4f}\n")
            f.write(f"- Side view: {fusion_results['fusion_results']['original_side_auroc']:.4f}\n\n")
        
        normal_front = merged_df.loc[~is_anomaly, 'score_front']
        anomaly_front = merged_df.loc[is_anomaly, 'score_front']
        normal_side = merged_df.loc[~is_anomaly, 'score_side']
        anomaly_side = merged_df.loc[is_anomaly, 'score_side']
        normal_fused = merged_df.loc[~is_anomaly, 'fused_score']
        anomaly_fused = merged_df.loc[is_anomaly, 'fused_score']
        
        f.write("Score Statistics:\n")
        f.write("- Front view (Normal): mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}\n".format(
            normal_front.mean(), normal_front.std(), normal_front.min(), normal_front.max()))
        f.write("- Front view (Anomaly): mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}\n".format(
            anomaly_front.mean(), anomaly_front.std(), anomaly_front.min(), anomaly_front.max()))
        f.write("- Side view (Normal): mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}\n".format(
            normal_side.mean(), normal_side.std(), normal_side.min(), normal_side.max()))
        f.write("- Side view (Anomaly): mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}\n".format(
            anomaly_side.mean(), anomaly_side.std(), anomaly_side.min(), anomaly_side.max()))
        f.write("- Fused {} (Normal): mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}\n".format(
            method, normal_fused.mean(), normal_fused.std(), normal_fused.min(), normal_fused.max()))
        f.write("- Fused {} (Anomaly): mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}\n".format(
            method, anomaly_fused.mean(), anomaly_fused.std(), anomaly_fused.min(), anomaly_fused.max()))
    
    return {
        'output_dir': output_dir,
        'figures': [
            'score_distributions.png',
            'front_vs_side_scores.png',
            'roc_curves.png',
            'auroc_comparison.png',
            'score_boxplots.png'
        ],
        'summary_file': 'analysis_summary.txt'
    }